In [ ]:
# Install required library
# pandas reads CSVs, deltalake. writes Delta tables
%pip install pandas deltalake


In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os

# Paths
BASE_PATH = "/lakehouse/default/Files/raw_ingestion"

print("Libraries loaded ")
print(f"Files available: {os.listdir(BASE_PATH)}")



In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os

# Run Bronze ingestion to correct location
SOURCE_FILES = {
    "patients":           "patients.csv",
    "encounters":         "encounters.csv",
    "providers":          "providers.csv",
    "claims_and_billing": "claims_and_billing.csv",
    "denials":            "denials.csv",
    "diagnoses":          "diagnoses.csv",
    "procedures":         "procedures.csv",
    "medications":        "medications.csv",
    "lab_tests":          "lab_tests.csv"
}

RAW_PATH = "/lakehouse/default/Files/raw_ingestion"
BRONZE_PATH = "/lakehouse/default/Files/bronze"

for table_name, file_name in SOURCE_FILES.items():
    print(f" {file_name}...")

    # Read CSV
    df = pd.read_csv(f"{RAW_PATH}/{file_name}")

    # Add audit columns
    df["_bronze_load_timestamp"] = datetime.now().isoformat()
    df["_source_file"] = file_name

    # Write to correct Bronze path
    output_path = f"{BRONZE_PATH}/{table_name}"
    os.makedirs(output_path, exist_ok=True)
    pq.write_table(
        pa.Table.from_pandas(df),
        f"{output_path}/part-0.parquet"
    )
    print(f" {table_name} — {len(df):,} rows")

print("\n Bronze ingestion complete!")


 patients.csv...
 patients — 60,000 rows
 encounters.csv...
 encounters — 70,000 rows
 providers.csv...
 providers — 1,491 rows
 claims_and_billing.csv...
 claims_and_billing — 70,000 rows
 denials.csv...
 denials — 5,998 rows
 diagnoses.csv...
 diagnoses — 70,000 rows
 procedures.csv...
 procedures — 126,021 rows
 medications.csv...
 medications — 94,498 rows
 lab_tests.csv...
 lab_tests — 54,537 rows

 Bronze ingestion complete!


In [ ]:
#Bronze Layer Verification
import os

print("=" * 50)
print(f"{'Table':<35} {'Rows':>12}")
print("=" * 50)

for table_name in SOURCE_FILES.keys():
    path = f"/lakehouse/default/Tables/bronze_{table_name}"
    df_check = pd.read_parquet(f"{path}/part-0.parquet")
    print(f"{'bronze_'+table_name:<35} {len(df_check):>12,}")

print("=" * 50)
print(" Bronze verification complete!")


Table                                       Rows
bronze_patients                           60,000
bronze_encounters                         70,000
bronze_providers                           1,491
bronze_claims_and_billing                 70,000
bronze_denials                             5,998
bronze_diagnoses                          70,000
bronze_procedures                        126,021
bronze_medications                        94,498
bronze_lab_tests                          54,537
 Bronze verification complete!


In [ ]:
# Verify we can see Bronze files
import os

print("Bronze files found:")
for folder in sorted(os.listdir("/lakehouse/default/Files")):
    print(f"  {folder}")


Bronze files found:
  bronze
  raw_ingestion
